# P135 — El sistema Hearsay-II: integrar conocimiento para resolver incertidumbre

## 1. Título y paper

**Paper:** *The Hearsay-II Speech-Understanding System: Integrating Knowledge to Resolve Uncertainty*  
**Autoría:** Lee D. Erman, Frederick Hayes-Roth, Victor R. Lesser, D. Raj Reddy  
**Año y venue:** 1980 · ACM Computing Surveys, 12(2), 213–253  
**Nivel:** L2 · **Motor:** `pizarra`  
**Ficha completa:** [`P135_pizarra`](../../papers/foundational/P135_pizarra/README.md)

**Hito:** Introduce la arquitectura de pizarra: fuentes de conocimiento independientes que publican hipótesis en una estructura compartida, sin llamarse entre sí.

- [doi:10.1145/356810.356816](https://doi.org/10.1145/356810.356816)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Entender habla exige combinar conocimiento acústico, léxico, sintáctico y semántico. Ninguna fuente decide sola, y encadenarlas en una tubería fija obliga a comprometerse pronto: un error temprano llega intacto al final.
2. Ejecutar una implementación mínima de la propuesta: Una estructura compartida —la pizarra— donde cada fuente escribe hipótesis parciales con su credibilidad, y un control oportunista que decide a quién invocar según lo que ya hay escrito. Nadie se compromete hasta que hay evidencia.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P58


## 4. Intuición

Cuatro fuentes de conocimiento y ninguna resuelve la frase sola. Publicando todas sus hipótesis donde las demás leen, la respuesta aparece — y no está en ninguna de ellas.


## 5. Concepto mínimo

```text
Tubería : fuente A decide → fuente B decide → ...   (compromiso temprano)
Pizarra : todas escriben hipótesis → nada se decide hasta el final

control OPORTUNISTA: a quién invocar depende de lo que ya hay escrito
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('pizarra', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántas posiciones resuelve la mejor fuente sola?
2. ¿Y todas juntas sobre la pizarra?
3. ¿Qué reconstruye una tubería de orden fijo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('pizarra', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('pizarra', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La mejor fuente sola resuelve **1 de 4** posiciones. Sobre la pizarra se resuelven **4 de 4** y la frase es «el gato come pescado», correcta. Una tubería donde cada etapa se compromete reconstruye «el gato come **pesado**» — 3 de 4.


## 10. Comentario pedagógico

Ese error es el punto entero. La etapa acústica no puede distinguir «pesado» de «pescado», así que decide mal; y aunque la etapa semántica sabría corregirlo, ya no puede. En la pizarra nadie decide hasta que todas han escrito, y por eso la evidencia débil de cuatro fuentes vale más que la decisión firme de una.


## 11. Error o anti-patrón deliberado

Anti-patrón: encadenar agentes especializados en una tubería fija.


In [ ]:
print('Cada eslabon se compromete con su mejor respuesta y el siguiente la hereda.')
print('Un error temprano llega intacto al final aunque alguien despues supiera corregirlo.')
print('Con memoria compartida, la decision se pospone hasta que hay evidencia.')

## 12. Corrección

Las dos arquitecturas, con el mismo conocimiento:


In [ ]:
r = run_paper_lab('pizarra', seed=3)['result']
for f in r['cada_fuente_por_separado']:
    print(f['fuente'], '->', f['posiciones_resueltas'], 'de', f['de'])
print('pizarra:', r['pizarra_compartida']['reconstruye'])
print('tuberia:', r['tuberia_de_orden_fijo']['reconstruye'])

## 13. Desafío guiado

Explica qué gana un sistema de pizarra al añadir una fuente nueva, comparado con lo que costaría añadirla a una tubería.


In [ ]:
r = run_paper_lab('pizarra', seed=3)['result']
show(r)

## 14. Desafío autónomo

Dibuja el flujo de un sistema multiagente tuyo. Marca dónde cada agente se compromete con una respuesta que otro podría corregir después y no puede.


## 15. Evidencia de aprendizaje

Guarda el diagrama y los puntos de compromiso temprano que encontraste.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P135_pizarra/README.md) · evaluación formal: [`assessments/papers/P135_pizarra.md`](../../assessments/papers/P135_pizarra.md)


## 16. Cierre

Si nadie manda, hay que repartir el trabajo de otra forma. Eso es P136.


## 17. Conexión con el siguiente hito

- P138

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
